## Import Packages and Mount Drive

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

.

.


## Load Dataset and Convert into Spectrograms
- Same as ML2-1

In [ ]:
!git clone https://github.com/ljwg3000/UNT_MEEN.git

In [ ]:
NoOfData = 180

for i in range(NoOfData):

    temp_path1 = f'/content/UNT_MEEN/AI_tutorial/Dataset/Normal_{i+1}'
    temp_path2 = f'/content/UNT_MEEN/AI_tutorial/Dataset/Abnormal_{i+1}'

    exec(f"Normal_{i+1}   = pd.read_csv(temp_path1 , sep=',' , header=None)")
    exec(f"Abnormal_{i+1} = pd.read_csv(temp_path2 , sep=',' , header=None)")

In [ ]:
DataLength = len(Normal_1)

AccData_Nor = pd.DataFrame(np.zeros((NoOfData, DataLength)))
AccData_Abn = pd.DataFrame(np.zeros((NoOfData, DataLength)))

for i in range(NoOfData):
  exec(f"tempNormal   = Normal_{i+1}")
  exec(f"tempAbnormal = Abnormal_{i+1}")

  AccData_Nor.iloc[i,:] = tempNormal.iloc[:,1]
  AccData_Abn.iloc[i,:] = tempAbnormal.iloc[:,1]

AccData = np.array(pd.concat([AccData_Nor, AccData_Abn], axis=0))
AccData.shape

In [ ]:
from scipy import signal

Fs = 12800  # Sampling Frequency
f,t,AccSTFT = signal.spectrogram(AccData, Fs, nperseg = 78, noverlap = 10)
AccSTFT.shape

In [ ]:
idx = 1  # Select index (1~180)

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.pcolormesh(t, f, AccSTFT[idx-1], cmap='jet')
plt.title(f"STFT (Normal_{idx})", fontsize=15)
plt.xlabel('Time(s)', fontsize=12)
plt.ylabel('Frequency(Hz)', fontsize=12)
plt.colorbar()

plt.subplot(1,2,2)
plt.pcolormesh(t, f, AccSTFT[idx+NoOfData-1], cmap='jet')
plt.title(f"STFT (Abnormal_{idx})", fontsize=15)
plt.xlabel('Time(s)', fontsize=12)
plt.colorbar()

plt.show()

.

.

## Prepare Training/Test Dataset and Labels
- Same as ML2-1

In [ ]:
NormalSet   = AccSTFT[:NoOfData]
AbnormalSet = AccSTFT[NoOfData:]

NoOfSensor  = 1
NormalSet   = NormalSet.reshape(NormalSet.shape[0], NormalSet.shape[1], NormalSet.shape[2], NoOfSensor)
AbnormalSet = AbnormalSet.reshape(AbnormalSet.shape[0], AbnormalSet.shape[1], AbnormalSet.shape[2], NoOfSensor)

NormalSet.shape, AbnormalSet.shape

In [ ]:
from sklearn.model_selection    import train_test_split

# Designate test data ratio
TestData_Ratio = 0.2

TrainData_Nor, TestData_Nor = train_test_split(NormalSet  , test_size=TestData_Ratio, random_state=777)
TrainData_Abn, TestData_Abn = train_test_split(AbnormalSet, test_size=TestData_Ratio, random_state=777)

print(TrainData_Nor.shape, TestData_Nor.shape)
print(TrainData_Abn.shape, TestData_Abn.shape)

In [ ]:
TrainLabel_Nor = np.zeros((TrainData_Nor.shape[0],2))
TrainLabel_Abn = np.ones( (TrainData_Abn.shape[0],2))
TestLabel_Nor  = np.zeros((TestData_Nor.shape[0],2))
TestLabel_Abn  = np.ones( (TestData_Abn.shape[0],2))

TrainLabel_Nor[:,0] = 1  # [1,0]: Normal
TrainLabel_Abn[:,0] = 0  # [0,1]: Abnormal
TestLabel_Nor[:,0]  = 1  # [1,0]: Normal
TestLabel_Abn[:,0]  = 0  # [0,1]: Abnormal

print(TrainLabel_Nor.shape, TestLabel_Nor.shape)
print(TrainLabel_Abn.shape, TestLabel_Abn.shape)

In [ ]:
TrainData  = np.concatenate([TrainData_Nor , TrainData_Abn ], axis=0)
TestData   = np.concatenate([TestData_Nor  , TestData_Abn  ], axis=0)
TrainLabel = np.concatenate([TrainLabel_Nor, TrainLabel_Abn], axis=0)
TestLabel  = np.concatenate([TestLabel_Nor , TestLabel_Abn ], axis=0)

print(TrainData.shape,  TestData.shape)
print(TrainLabel.shape, TestLabel.shape)

.

.

.


## Load CNN model
- Load the model we've saved in ML2-1

In [ ]:
model = keras.models.load_model('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/CNN_model.keras')
model.summary()

In [ ]:
Loss, Accuracy = model.evaluate(TestData, TestLabel, verbose=0)
print('[Performance of CNN model] \n')
print('Accuracy : {:.2f}%'.format(Accuracy*100))

.

.

.

# Explainabla AI Practice: Integrated Gradients (IG)

### 1️⃣ `show_input_and_heatmap()`  
This function **visualizes** both the input spectrogram and its explanation (Grad-CAM or Integrated Gradients).  
- We flip the image vertically (`np.flipud`) so that **low frequencies appear at the bottom**, which matches how spectrograms are usually shown.  

- When a heatmap is provided, it plots **two side-by-side images**:  
  - The original spectrogram  
  - The explanation heatmap (colored intensity = importance)  
This helps us *see which regions the model focused on* while making its prediction.

In [ ]:
def show_input_and_heatmap(x_img, heatmap, title="", cmap="viridis"):

    fig, ax = plt.subplots(1, 2, figsize=(6,3))
    x_img = np.flipud(x_img)
    heatmap = np.flipud(heatmap)

    ax[0].imshow(x_img.squeeze(), cmap="gray")
    ax[0].set_title("input")
    ax[0].axis("off")
    im = ax[1].imshow(heatmap, cmap=cmap)
    ax[1].set_title(title)
    ax[1].axis("off")
    # fig.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

### 2️⃣ `compute_gradients()`  
This function computes the **gradient of the model’s output with respect to the input spectrogram**.  
- It uses TensorFlow’s `GradientTape()` to track how the output changes when the input changes.  
- For the chosen target class (e.g., “fault”), it calculates  

  $\frac{\partial y_{\text{target}}}{\partial x}$
  
- The result tells us **how sensitive each pixel (frequency-time cell)** is to the target class output —  
  higher gradients mean those regions influence the prediction more.

In [ ]:
def compute_gradients(model, x, target_index):

    x = tf.convert_to_tensor(x)
    with tf.GradientTape() as tape:
        tape.watch(x)
        preds  = model(x, training=False)
        target = preds[:, target_index]
    grads = tape.gradient(target, x)

    return grads

### 3️⃣ `integrated_gradients()`  
This function implements **Integrated Gradients (IG)**, a method to attribute importance to each pixel in the input.  
- Instead of computing just one gradient, IG averages gradients **along a straight line path** from a baseline (usually all zeros) to the actual input.  
- It divides the path into small steps, computes gradients at each step, then integrates (averages) them.  
- Finally, it multiplies the average gradient by the difference between the input and baseline.  
- The result is a **heatmap** showing which areas of the spectrogram contributed the most to the model’s decision.  
This approach is smoother and more stable than using a single gradient.

In [ ]:
def integrated_gradients(model, x, target_index, baseline=None, steps=32):
    """
    Compute Integrated Gradients for a given input.
    baseline: reference input (default zeros)
    steps: number of interpolation steps
    return: (40,40) normalized attribution map
    """
    if baseline is None:
        baseline = np.zeros_like(x, dtype=np.float32)

    scaled_inputs = [baseline + (i/steps)*(x - baseline) for i in range(1, steps+1)]
    scaled_inputs = np.concatenate(scaled_inputs, axis=0)  # (steps,40,40,1)

    grads = []
    for i in range(steps):
        g = compute_gradients(model, scaled_inputs[i:i+1], target_index)
        grads.append(g.numpy())
    avg_grads = np.mean(np.concatenate(grads, axis=0), axis=0, keepdims=True)  # (1,40,40,1)

    ig = (x - baseline) * avg_grads
    ig = ig[0]
    ig = np.mean(ig, axis=-1)  # average over channels

    ig = ig - ig.min()
    if ig.max() > 0:
        ig = ig / (ig.max() + 1e-8)
    return ig

### 4️⃣ Main Visualization Loop  
In the last code cell:  
- We randomly select a few samples from the test data.  
- For each sample, the model predicts whether it’s *normal* or *fault*.  
- We then compute and display the **Integrated Gradients heatmap** (or Grad-CAM).  
- The titles show the **predicted class**, the **confidence score**, and the **true label**.  

By comparing the *input* and *heatmap* images, we can visually confirm:
- Which parts of the spectrogram most influenced the model’s prediction.  
- Whether those areas correspond to meaningful physical features (e.g., specific frequency bands related to faults).  

In [ ]:
idxs = np.random.choice(TestData.shape[0], size=4, replace=False)

CLASS_NAMES = ["normal", "fault"]
for idx in idxs:
    x = TestData[idx:idx+1]
    y_true = np.argmax(TestLabel[idx])
    p = model.predict(x, verbose=0)[0]
    y_pred = int(np.argmax(p))
    conf = float(p[y_pred])
    title_pred = f"pred={CLASS_NAMES[y_pred]} ({conf:.2f}), true={CLASS_NAMES[y_true]}"

    ig = integrated_gradients(model, x, target_index=y_pred, baseline=None, steps=32)
    show_input_and_heatmap(x[0], ig, title="Integrated Gradients\n"+title_pred, cmap="magma")
    print('\n')
